# Check `GDS` Python stack

This notebook checks all software requirements for the course Geographic Data Science are correctly installed. 

A successful run of this notebook implies no errors returned in any cell. This ensures a correct environment installed.

**There could still be a bunch of warnings thrown, for example in red cells, but as long as you arrive at the last cell all is fine!**

## Imports

In [ ]:
import black

In [ ]:
import bokeh

In [ ]:
import boto3

In [ ]:
import bottleneck

In [ ]:
#Issues with the Census API
import traceback
try:
    import cenpy
except Exception:
    traceback.print_exc()

In [ ]:
import clustergram

In [ ]:
import contextily

In [ ]:
import cython

In [ ]:
import dask

In [ ]:
import dask_geopandas

In [ ]:
import datashader

In [ ]:
import flake8

In [ ]:
import geocube

In [ ]:
import geopandas

In [ ]:
import geopy

In [ ]:
import h3

In [ ]:
import hdbscan

In [ ]:
import pystac_client

In [ ]:
import ipyleaflet

In [ ]:
import ipympl

In [ ]:
import ipywidgets

In [ ]:
import legendgram

In [ ]:
import lxml

In [ ]:
import momepy

In [ ]:
import netCDF4

In [ ]:
import networkx

In [ ]:
import osmnx

In [ ]:
import palettable

In [ ]:
import pandana

In [ ]:
import polyline

In [ ]:
import psycopg2

In [ ]:
import pyarrow

In [ ]:
import pygeos

In [ ]:
import pyrosm

In [ ]:
import pysal
import pysal.lib
import pysal.explore
import pysal.model
import pysal.viz

In [ ]:
import rasterio

In [ ]:
import rasterstats

In [ ]:
import rio_cogeo

In [ ]:
import rioxarray

In [ ]:
import sklearn

In [ ]:
import seaborn

In [ ]:
import spatialpandas

In [ ]:
import sqlalchemy

In [ ]:
import statsmodels

In [ ]:
import tabulate

In [ ]:
import urbanaccess

In [ ]:
import xarray_leaflet

In [ ]:
import xrspatial

In [ ]:
import xlrd

In [ ]:
import xlsxwriter

---

`pip` installs:

In [ ]:
import pooch

In [ ]:
import geoalchemy2

In [ ]:
import matplotlib_scalebar

In [ ]:
import pygeoda

In [ ]:
import pytest_cov

In [ ]:
import pytest_tornasync

In [ ]:
#Does not support Windows/Python3.9 for now#import simplification

In [ ]:
import topojson

In [ ]:
# This is broken at point of release because of:#https://github.com/urbangrammarai/graphics/issues/2#import urbangrammar_graphics

---

**Legacy checks** (in some ways superseded by those above but in some still useful)

In [ ]:
import bokeh as bk
float(bk.__version__[:1]) >= 1

In [ ]:
import matplotlib as mpl
float(mpl.__version__[:3]) >= 1.5

In [ ]:
import seaborn as sns
float(sns.__version__[:3]) >= 0.6

In [ ]:
import datashader as ds
float(ds.__version__[:3]) >= 0.6

In [ ]:
import palettable as pltt
float(pltt.__version__[:3]) >= 3.1

In [ ]:
sns.palplot(pltt.matplotlib.Viridis_10.hex_colors)

---

In [ ]:
import pandas as pd
float(pd.__version__[:3]) >= 1

In [ ]:
import dask
float(dask.__version__[:1]) >= 1

In [ ]:
import statsmodels.api as sm
float(sm.__version__[2:4]) >= 10

---

In [ ]:
import fiona
float(fiona.__version__[:3]) >= 1.8

In [ ]:
import geopandas as gpd
float(gpd.__version__[:3]) >= 0.4

In [ ]:
import pysal as ps
float(ps.__version__[:1]) >= 2

In [ ]:
import rasterio as rio
float(rio.__version__[:1]) >= 1

## Hardcoded tests

In [ ]:
from pysal import lib as libps
shp = libps.examples.get_path('columbus.shp')
db = geopandas.read_file(shp)
db.head()

In [ ]:
db[['AREA', 'PERIMETER']].to_feather('db.feather')
tst = pd.read_feather('db.feather')
! rm db.feather

In [ ]:
db.to_parquet('db.pq')
tst = gpd.read_parquet('db.pq')
! rm db.pq

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
f, ax = plt.subplots(1)
db.plot(facecolor='yellow', ax=ax)
ax.set_axis_off()
plt.show()

In [ ]:
db.crs = 'EPSG:26918'

In [ ]:
db_wgs84 = db.to_crs(epsg=4326)
db_wgs84.plot()
plt.show()

In [ ]:
from pysal.viz import splot
from splot.mapping import vba_choropleth

f, ax = vba_choropleth(db['INC'], db['HOVAL'], db)

---

In [ ]:
db.plot(column='INC', scheme='fisher_jenks', cmap=plt.matplotlib.cm.Blues)
plt.show()

In [ ]:
city = osmnx.geocode_to_gdf('Berkeley, California, US')
osmnx.plot_footprints(osmnx.project_gdf(city));

In [ ]:
from pyrosm import get_data
# Download data for the city of Helsinki
fp = get_data("Helsinki", directory="./")
print(fp)

# Get filepath to test PBF dataset
fp = pyrosm.get_data("test_pbf")
print("Filepath to test data:", fp)

# Initialize the OSM object
osm = pyrosm.OSM(fp)

# See the type
print("Type of 'osm' instance: ", type(osm))

from pyrosm import get_data

# Pyrosm comes with a couple of test datasets
# that can be used straight away without
# downloading anything
fp = get_data("test_pbf")

# Initialize the OSM parser object
osm = pyrosm.OSM(fp)

# Read all drivable roads
# =======================
drive_net = osm.get_network(network_type="driving")
drive_net.plot()

! rm Helsinki.osm.pbf

---

In [ ]:
import numpy as np
import contextily as ctx
tl = ctx.providers.CartoDB.Positron

db = geopandas.read_file(ps.lib.examples.get_path('us48.shp'))
db.crs = "EPSG:4326"
dbp = db.to_crs(epsg=3857)
w, s, e, n = dbp.total_bounds
# Download raster
_ = ctx.bounds2raster(w, s, e, n, 'us.tif', source=tl)
# Load up and plot
source = rio.open('us.tif', 'r')
red = source.read(1)
green = source.read(2)
blue = source.read(3)
pix = np.dstack((red, green, blue))
bounds = (source.bounds.left, source.bounds.right, \
          source.bounds.bottom, source.bounds.top)
f = plt.figure(figsize=(6, 6))
ax = plt.imshow(pix, extent=bounds)

In [ ]:
from ipyleaflet import Map, basemaps, basemap_to_tiles, SplitMapControl

m = Map(center=(42.6824, 365.581), zoom=5)

right_layer = basemap_to_tiles(basemaps.NASAGIBS.ModisTerraTrueColorCR, "2017-11-11")
left_layer = basemap_to_tiles(basemaps.NASAGIBS.ModisAquaBands721CR, "2017-11-11")

control = SplitMapControl(left_layer=left_layer, right_layer=right_layer)
m.add_control(control)

m

In [ ]:
# Convert us.tiff to COG with rio-cogeo
! rio cogeo create us.tif us_cog.tif
! rio cogeo validate us_cog.tif
! rio info us_cog.tif
! rio warp --dst-crs EPSG:4326 us_cog.tif us_cog_ll.tif

In [ ]:
# rioxarray + xarray_leaflet
import xarray
us = rioxarray.open_rasterio(
    "us_cog.tif"
).sel(
    band=1, 
    y=slice(7000000, 5000000),
    x=slice(-10000000, -8000000)
)
import xarray_leaflet
from ipyleaflet import Map

m = Map(zoom=1, basemap=basemaps.CartoDB.DarkMatter)
m

In [ ]:
l = us.astype(
    float
).rio.reproject(
    "EPSG:4326"
).leaflet.plot(m)
l.interact(opacity=(0.0,1.0))

In [ ]:
! rm us.tif us_cog.tif us_cog_ll.tif
! rm -rf cache/

In [ ]:
from IPython.display import GeoJSON

GeoJSON({
    "type": "Feature",
    "geometry": {
        "type": "Point",
        "coordinates": [-118.4563712, 34.0163116]
    }
})

In [ ]:
print("All checks passed - Have fun in class!")